# Convex Ball-DP end-to-end worked example

This notebook gives one theorem-aligned, end-to-end example for the convex Gaussian-output path in `quantbayes.ball_dp`.

It assumes that `X_train`, `X_test`, `y_train`, and `y_test` already exist in memory.

The notebook does exactly one worked example:

1. prepare the embedding arrays and integer labels;
2. choose a Ball radius from within-label distances;
3. fit a Ball-DP release and a matched-privacy standard-DP comparator;
4. report noise scale, sensitivity, and test accuracy;
5. build one finite same-label candidate set inside the chosen Ball;
6. compute exact-identification reconstruction-robustness bounds (direct Gaussian and RDP);
7. run the exact finite-prior Bayes attack and compare it against the oblivious baseline `1/m`.

The standard comparator is obtained by using radius `2 * EMBEDDING_BOUND` at the same `(epsilon, delta)`.

In [ ]:
import math
from dataclasses import dataclass
from typing import Optional

import jax
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from quantbayes.ball_dp import (
    attack_convex_ball_output_finite_prior,
    ball_rero,
    fit_convex,
    make_finite_identification_prior,
    select_ball_radius,
    summarize_embedding_ball_radii,
)

plt.rcParams.update(
    {
        "figure.figsize": (7.2, 4.8),
        "figure.dpi": 140,
        "savefig.dpi": 220,
        "font.size": 11,
        "axes.titlesize": 13,
        "axes.labelsize": 11,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "grid.linestyle": "--",
        "legend.frameon": False,
        "lines.linewidth": 2.2,
        "lines.markersize": 7,
    }
)

BALL_COLOR = "#1f77b4"
STANDARD_COLOR = "#d62728"
NEUTRAL_COLOR = "#6b7280"
ACCENT_COLOR = "#2ca02c"

np.set_printoptions(precision=4, suppress=True)
print("JAX backend:", jax.default_backend())
print("JAX devices:", jax.devices())

## 1. Data contract

This notebook assumes that `X_train`, `X_test`, `y_train`, and `y_test` are already defined.

The preparation cell below converts them to the shapes expected by `fit_convex(...)`:

- features are cast to `float32` and reshaped to `[n, d]` if needed;
- labels are integer-encoded to `0, 1, ..., K-1`;
- binary heads (`binary_logistic`, `sqr_hinge_svm`) therefore receive labels in `{0, 1}`.

In [ ]:
def _as_2d_float32(X, name: str) -> np.ndarray:
    X = np.asarray(X, dtype=np.float32)
    if X.ndim < 2:
        raise ValueError(f"{name} must have shape [n, d] or [n, ...].")
    return X.reshape(len(X), -1)


def _encode_labels_train_test(y_train_raw, y_test_raw):
    y_train_raw = np.asarray(y_train_raw)
    y_test_raw = np.asarray(y_test_raw)
    all_labels = np.concatenate([y_train_raw.reshape(-1), y_test_raw.reshape(-1)], axis=0)
    unique_labels = list(pd.Index(all_labels).unique())
    label_to_index = {label: idx for idx, label in enumerate(unique_labels)}
    y_train_enc = np.asarray([label_to_index[v] for v in y_train_raw.reshape(-1)], dtype=np.int32)
    y_test_enc = np.asarray([label_to_index[v] for v in y_test_raw.reshape(-1)], dtype=np.int32)
    return unique_labels, label_to_index, y_train_enc, y_test_enc


X_train = _as_2d_float32(X_train, "X_train")
X_test = _as_2d_float32(X_test, "X_test")
original_label_values, label_to_index, y_train, y_test = _encode_labels_train_test(y_train, y_test)

if X_train.shape[1] != X_test.shape[1]:
    raise ValueError("X_train and X_test must have the same feature dimension.")

num_classes = int(len(np.unique(y_train)))
train_norms = np.linalg.norm(X_train, axis=1)
test_norms = np.linalg.norm(X_test, axis=1)

summary_df = pd.DataFrame(
    {
        "split": ["train", "test"],
        "n_examples": [len(X_train), len(X_test)],
        "dimension": [X_train.shape[1], X_test.shape[1]],
        "num_classes": [num_classes, num_classes],
        "max_l2_norm": [float(train_norms.max()), float(test_norms.max())],
        "mean_l2_norm": [float(train_norms.mean()), float(test_norms.mean())],
    }
)

display(summary_df.round(6))
print("Original label values -> encoded labels:")
for label, idx in label_to_index.items():
    print(f"  {label!r} -> {idx}")

## 2. Configuration

Edit only this cell if you want to change the model family or the single worked-out configuration.

The notebook accepts the convex families

- `ridge_prototype`
- `softmax_logistic`
- `binary_logistic`
- `sqr_hinge_svm`

with `sqr_hinge_svm` normalized to the repo’s canonical `squared_hinge` name.

In [ ]:
def canonicalize_model_family(name: str) -> str:
    key = str(name).strip().lower().replace("-", "_")
    mapping = {
        "ridge_prototype": "ridge_prototype",
        "softmax_logistic": "softmax_logistic",
        "binary_logistic": "binary_logistic",
        "sqr_hinge_svm": "squared_hinge",
        "squared_hinge": "squared_hinge",
    }
    if key not in mapping:
        raise ValueError(
            "MODEL_FAMILY must be one of {'ridge_prototype', 'softmax_logistic', 'binary_logistic', 'sqr_hinge_svm'}."
        )
    return mapping[key]


RADIUS_TAG_TO_QUANTILE = {"q50": 0.50, "q80": 0.80, "q95": 0.95}

MODEL_FAMILY = canonicalize_model_family("ridge_prototype")
RADIUS_TAG = "q80"
EPSILON = 1.0
DELTA = 1e-6
EMBEDDING_BOUND = 1.0
LAM = 1e-2
MAX_ITER = 100
SOLVER = "lbfgs_fullbatch"
SEED = 0
ORDERS = tuple(list(range(2, 65)) + [80, 96, 128, 160, 192, 256])

ATTACK_M = 16
ATTACK_ETA = 0.5  # any value in [0, 1) is equivalent for exact identification under the finite prior
PREFERRED_TARGET_INDEX: Optional[int] = None
TARGET_SEARCH_SEED = 7
CANDIDATE_DRAW_SEED = 13
USE_RANDOM_CANDIDATE_DRAW = True

if RADIUS_TAG not in RADIUS_TAG_TO_QUANTILE:
    raise ValueError("RADIUS_TAG must be one of {'q50', 'q80', 'q95'}." )
if MODEL_FAMILY in {"binary_logistic", "squared_hinge"} and num_classes != 2:
    raise ValueError(
        f"MODEL_FAMILY={MODEL_FAMILY!r} is binary-only, but the dataset has {num_classes} classes."
    )
if ATTACK_M < 2:
    raise ValueError("ATTACK_M must be at least 2.")

print({
    "MODEL_FAMILY": MODEL_FAMILY,
    "RADIUS_TAG": RADIUS_TAG,
    "EPSILON": EPSILON,
    "DELTA": DELTA,
    "EMBEDDING_BOUND": EMBEDDING_BOUND,
    "LAM": LAM,
    "MAX_ITER": MAX_ITER,
    "ATTACK_M": ATTACK_M,
})

train_max_norm = float(train_norms.max())
if train_max_norm > EMBEDDING_BOUND + 1e-6:
    print(
        f"Warning: max ||x|| on the training split is {train_max_norm:.6f}, which exceeds EMBEDDING_BOUND={EMBEDDING_BOUND:.6f}."
    )
    print("The standard comparator radius = 2 * EMBEDDING_BOUND is theorem-faithful only if EMBEDDING_BOUND is a valid public norm bound.")

## 3. Radius selection

We choose the Ball radius from within-label pairwise distances using the public API:

- `summarize_embedding_ball_radii(...)`
- `select_ball_radius(...)`

By default this notebook uses the conservative `max_labelwise` quantile rule, exactly as in the experiment scripts.

In [ ]:
radius_report = summarize_embedding_ball_radii(
    X_train,
    y_train,
    quantiles=(0.50, 0.80, 0.95, 1.00),
    max_exact_pairs=250_000,
    max_sampled_pairs=100_000,
    seed=SEED,
)

radius = float(
    select_ball_radius(
        radius_report,
        strategy="max_labelwise_quantile",
        quantile=RADIUS_TAG_TO_QUANTILE[RADIUS_TAG],
    )
)
standard_radius = float(2.0 * EMBEDDING_BOUND)

radius_table = pd.DataFrame(
    [
        {
            "radius_name": "q50",
            "value": float(radius_report["max_labelwise_quantiles"]["q0.5"]),
        },
        {
            "radius_name": "q80",
            "value": float(radius_report["max_labelwise_quantiles"]["q0.8"]),
        },
        {
            "radius_name": "q95",
            "value": float(radius_report["max_labelwise_quantiles"]["q0.95"]),
        },
        {
            "radius_name": "within-label max",
            "value": float(radius_report["global_max_within_label_distance_observed"]),
        },
        {
            "radius_name": "standard radius = 2B",
            "value": standard_radius,
        },
    ]
)

display(radius_table.round(6))
print(f"Chosen Ball radius ({RADIUS_TAG}) = {radius:.6f}")
print(f"Standard comparator radius     = {standard_radius:.6f}")

fig, ax = plt.subplots(figsize=(7.6, 4.4))
ax.bar(radius_table["radius_name"], radius_table["value"], color=[NEUTRAL_COLOR, BALL_COLOR, ACCENT_COLOR, STANDARD_COLOR, STANDARD_COLOR])
ax.axhline(radius, color=BALL_COLOR, linestyle="--", linewidth=2, label=f"chosen Ball radius ({RADIUS_TAG})")
ax.axhline(standard_radius, color=STANDARD_COLOR, linestyle=":", linewidth=2, label="standard radius = 2B")
ax.set_ylabel("radius")
ax.set_title("Radius choices from within-label geometry")
ax.tick_params(axis="x", rotation=18)
ax.legend()
plt.show()

## 4. Fit the matched-privacy releases

We fit two releases at the same `(epsilon, delta)`:

- **Ball-DP** with the selected radius `r`;
- **standard DP comparator** with radius `2 * EMBEDDING_BOUND`.

For the standard comparator we still call `fit_convex(..., privacy="ball_dp")`; the only change is the larger radius. This matches the experiment design in the paper.

In [ ]:
def fit_release(radius_value: float, *, seed: int):
    return fit_convex(
        X_train,
        y_train,
        X_eval=X_test,
        y_eval=y_test,
        model_family=MODEL_FAMILY,
        privacy="ball_dp",
        radius=float(radius_value),
        lam=float(LAM),
        epsilon=float(EPSILON),
        delta=float(DELTA),
        embedding_bound=float(EMBEDDING_BOUND),
        num_classes=int(num_classes),
        orders=ORDERS,
        solver=SOLVER,
        max_iter=int(MAX_ITER),
        seed=int(seed),
    )


ball_release = fit_release(radius, seed=SEED)
standard_release = fit_release(standard_radius, seed=SEED)


def release_summary_row(name: str, release, effective_radius: float):
    cert = release.privacy.ball.dp_certificates[0] if release.privacy.ball.dp_certificates else None
    return {
        "mechanism": name,
        "radius": float(effective_radius),
        "delta_sensitivity": float(release.sensitivity.delta_ball),
        "sigma": float(release.privacy.ball.sigma),
        "accuracy": float(release.utility_metrics.get("accuracy", np.nan)),
        "epsilon_cert": None if cert is None else float(cert.epsilon),
        "delta_cert": None if cert is None else float(cert.delta),
        "lz": None if release.sensitivity.lz is None else float(release.sensitivity.lz),
        "lz_source": release.sensitivity.lz_source,
        "exact_vs_upper": release.sensitivity.exact_vs_upper,
    }


erg_df = pd.DataFrame(
    [
        release_summary_row("Ball-DP", ball_release, radius),
        release_summary_row("Standard-DP", standard_release, standard_radius),
    ]
)

display(erg_df.round(6))

fig, axes = plt.subplots(1, 2, figsize=(11.2, 4.0))
axes[0].bar(erg_df["mechanism"], erg_df["sigma"], color=[BALL_COLOR, STANDARD_COLOR])
axes[0].set_ylabel(r"noise scale $\sigma$")
axes[0].set_title("Matched-privacy noise scale")

axes[1].bar(erg_df["mechanism"], erg_df["accuracy"], color=[BALL_COLOR, STANDARD_COLOR])
axes[1].set_ylabel("test accuracy")
axes[1].set_title("Matched-privacy utility")

plt.tight_layout()
plt.show()

## 5. Build one finite feasible candidate set

For the attack demonstration we need a finite prior support

$$
S = \{z_1, \dots, z_m\}
$$

such that

- the true target is included;
- every candidate has the same label as the target;
- every candidate lies inside the Ball of radius `r` around the target.

The helper below searches for a target with at least `m - 1` same-label in-radius distractors, using both the train and test splits as the candidate pool.

In [ ]:
@dataclass
class CandidatePool:
    target_index: int
    target_label: int
    target_vector: np.ndarray
    pool_vectors: np.ndarray
    pool_source_ids: list[str]
    pool_distances: np.ndarray


def _deduplicate_vectors(vectors, source_ids, distances, *, exclude_value=None, atol=1e-7):
    kept_vectors = []
    kept_ids = []
    kept_dists = []
    for vec, src, dist in zip(vectors, source_ids, distances):
        vec = np.asarray(vec, dtype=np.float32)
        if exclude_value is not None and np.allclose(vec, exclude_value, atol=atol, rtol=0.0):
            continue
        if any(np.allclose(vec, old, atol=atol, rtol=0.0) for old in kept_vectors):
            continue
        kept_vectors.append(vec)
        kept_ids.append(str(src))
        kept_dists.append(float(dist))
    if not kept_vectors:
        return (
            np.zeros((0, X_train.shape[1]), dtype=np.float32),
            [],
            np.zeros((0,), dtype=np.float32),
        )
    return (
        np.stack(kept_vectors, axis=0).astype(np.float32),
        kept_ids,
        np.asarray(kept_dists, dtype=np.float32),
    )


def build_candidate_pool(target_index: int, radius_value: float) -> CandidatePool:
    x_target = np.asarray(X_train[target_index], dtype=np.float32)
    target_label = int(y_train[target_index])

    train_same = np.where(y_train == target_label)[0]
    train_same = train_same[train_same != int(target_index)]
    test_same = np.where(y_test == target_label)[0]

    train_vectors = np.asarray(X_train[train_same], dtype=np.float32)
    test_vectors = np.asarray(X_test[test_same], dtype=np.float32)

    train_dists = np.linalg.norm(train_vectors - x_target[None, :], axis=1) if len(train_vectors) else np.zeros((0,), dtype=np.float32)
    test_dists = np.linalg.norm(test_vectors - x_target[None, :], axis=1) if len(test_vectors) else np.zeros((0,), dtype=np.float32)

    keep_train = train_dists <= float(radius_value) + 1e-7
    keep_test = test_dists <= float(radius_value) + 1e-7

    pool_vectors = np.concatenate([train_vectors[keep_train], test_vectors[keep_test]], axis=0)
    pool_source_ids = [f"train:{int(i)}" for i in train_same[keep_train].tolist()] + [f"test:{int(i)}" for i in test_same[keep_test].tolist()]
    pool_distances = np.concatenate([train_dists[keep_train], test_dists[keep_test]], axis=0)

    pool_vectors, pool_source_ids, pool_distances = _deduplicate_vectors(
        pool_vectors,
        pool_source_ids,
        pool_distances,
        exclude_value=x_target,
    )

    return CandidatePool(
        target_index=int(target_index),
        target_label=int(target_label),
        target_vector=x_target,
        pool_vectors=pool_vectors,
        pool_source_ids=pool_source_ids,
        pool_distances=pool_distances,
    )


def choose_feasible_target(*, m: int, radius_value: float, preferred_index: Optional[int], seed: int) -> CandidatePool:
    if preferred_index is not None:
        pool = build_candidate_pool(int(preferred_index), radius_value)
        if len(pool.pool_vectors) >= int(m) - 1:
            return pool

    rng = np.random.default_rng(int(seed))
    for idx in rng.permutation(len(X_train)).tolist():
        pool = build_candidate_pool(int(idx), radius_value)
        if len(pool.pool_vectors) >= int(m) - 1:
            return pool

    raise RuntimeError(
        f"Could not find a target with at least {m - 1} same-label candidates inside radius {radius_value:.6f}."
    )


def draw_candidate_set(pool: CandidatePool, *, m: int, seed: int, random_draw: bool):
    need = int(m) - 1
    if len(pool.pool_vectors) < need:
        raise ValueError("Pool is too small for the requested candidate set size.")

    rng = np.random.default_rng(int(seed))
    if random_draw:
        chosen = rng.permutation(len(pool.pool_vectors))[:need]
    else:
        chosen = np.argsort(pool.pool_distances)[:need]

    distractors = np.asarray(pool.pool_vectors[chosen], dtype=np.float32)
    distractor_ids = [pool.pool_source_ids[int(i)] for i in chosen.tolist()]
    distractor_dists = np.asarray(pool.pool_distances[chosen], dtype=np.float32)

    X_candidates = np.concatenate([distractors, pool.target_vector[None, :]], axis=0).astype(np.float32)
    y_candidates = np.full((m,), int(pool.target_label), dtype=np.int32)
    candidate_source_ids = list(distractor_ids) + [f"target:{pool.target_index}"]
    candidate_distances = np.concatenate([distractor_dists, np.array([0.0], dtype=np.float32)], axis=0)

    order = rng.permutation(m)
    X_candidates = X_candidates[order]
    y_candidates = y_candidates[order]
    candidate_source_ids = [candidate_source_ids[int(i)] for i in order.tolist()]
    candidate_distances = candidate_distances[order]
    return X_candidates, y_candidates, candidate_source_ids, candidate_distances


pool = choose_feasible_target(
    m=ATTACK_M,
    radius_value=radius,
    preferred_index=PREFERRED_TARGET_INDEX,
    seed=TARGET_SEARCH_SEED,
)

X_candidates, y_candidates, candidate_source_ids, candidate_distances = draw_candidate_set(
    pool,
    m=ATTACK_M,
    seed=CANDIDATE_DRAW_SEED,
    random_draw=USE_RANDOM_CANDIDATE_DRAW,
)

attack_prior = make_finite_identification_prior(X_candidates, weights=None)
target_index = int(pool.target_index)
target_label = int(pool.target_label)
true_source_id = f"target:{target_index}"

candidate_df = pd.DataFrame(
    {
        "candidate_id": np.arange(len(X_candidates), dtype=int),
        "source_id": candidate_source_ids,
        "distance_to_target": candidate_distances,
        "is_true_target": [sid == true_source_id for sid in candidate_source_ids],
    }
).sort_values(["is_true_target", "distance_to_target"], ascending=[False, True]).reset_index(drop=True)

display(candidate_df.round(6))
print(f"chosen target_index = {target_index}")
print(f"target_label        = {target_label}")
print(f"candidate set size  = {len(X_candidates)}")
print(f"max candidate dist  = {float(candidate_distances.max()):.6f}")

## 6. Compute ReRo bounds and run the finite-prior exact Bayes attack

For the finite exact-identification prior, any `eta < 1` corresponds to the same 0/1 exact-identification event. We therefore use a single point `eta = 0.5`.

We keep **two comparison regimes separate**:

1. **Matched privacy**: compare the Ball release at radius `r` with a separate standard release at radius `2B`, both calibrated to the same `(epsilon, delta)`.
   For Gaussian releases, the direct bound depends on `Delta / sigma`. Because both the classical and analytic Gaussian calibrations are linear in sensitivity, the matched-privacy direct bounds should agree up to numerical tolerance.

2. **Same sigma (geometry only)**: evaluate the standard sensitivity at the Ball release's `sigma`.
   This is **not** a matched-privacy comparison. It isolates the effect of local vs global geometry at fixed noise.

The notebook therefore reports matched-privacy bounds and same-sigma geometry references in separate tables and separate panels.


In [ ]:
eta_grid = (float(ATTACK_ETA),)

ball_direct_report = ball_rero(ball_release, prior=attack_prior, eta_grid=eta_grid, mode="gaussian_direct")
ball_rdp_report = ball_rero(ball_release, prior=attack_prior, eta_grid=eta_grid, mode="rdp")

standard_direct_report = ball_rero(standard_release, prior=attack_prior, eta_grid=eta_grid, mode="gaussian_direct")
standard_rdp_report = ball_rero(standard_release, prior=attack_prior, eta_grid=eta_grid, mode="rdp")

ball_direct_point = ball_direct_report.points[0]
ball_rdp_point = ball_rdp_report.points[0]
standard_direct_point = standard_direct_report.points[0]
standard_rdp_point = standard_rdp_report.points[0]

same_sigma_standard_direct = float(ball_direct_point.gamma_standard)
same_sigma_standard_rdp = float(ball_rdp_point.gamma_standard)

ball_attack, _, _ = attack_convex_ball_output_finite_prior(
    ball_release,
    X_train,
    y_train,
    target_index=target_index,
    X_candidates=X_candidates,
    y_candidates=y_candidates,
    prior_weights=None,
    known_label=target_label,
    eta_grid=eta_grid,
)

standard_attack, _, _ = attack_convex_ball_output_finite_prior(
    standard_release,
    X_train,
    y_train,
    target_index=target_index,
    X_candidates=X_candidates,
    y_candidates=y_candidates,
    prior_weights=None,
    known_label=target_label,
    eta_grid=eta_grid,
)

baseline = 1.0 / float(len(X_candidates))

matched_bounds_df = pd.DataFrame(
    [
        {
            "bound_family": "direct",
            "Ball": float(ball_direct_point.gamma_ball),
            "Standard matched": float(standard_direct_point.gamma_ball),
        },
        {
            "bound_family": "RDP",
            "Ball": float(ball_rdp_point.gamma_ball),
            "Standard matched": float(standard_rdp_point.gamma_ball),
        },
    ]
)
matched_bounds_df["abs_gap"] = np.abs(matched_bounds_df["Ball"] - matched_bounds_df["Standard matched"])

same_sigma_bounds_df = pd.DataFrame(
    [
        {
            "bound_family": "direct",
            "Ball": float(ball_direct_point.gamma_ball),
            "Standard @ sigma_Ball": float(same_sigma_standard_direct),
        },
        {
            "bound_family": "RDP",
            "Ball": float(ball_rdp_point.gamma_ball),
            "Standard @ sigma_Ball": float(same_sigma_standard_rdp),
        },
    ]
)
same_sigma_bounds_df["gap_vs_Ball"] = np.abs(same_sigma_bounds_df["Ball"] - same_sigma_bounds_df["Standard @ sigma_Ball"])

consistency_df = pd.DataFrame(
    [
        {
            "check": "matched direct abs gap",
            "value": float(abs(ball_direct_point.gamma_ball - standard_direct_point.gamma_ball)),
        },
        {
            "check": "matched RDP abs gap",
            "value": float(abs(ball_rdp_point.gamma_ball - standard_rdp_point.gamma_ball)),
        },
    ]
)

attack_df = pd.DataFrame(
    [
        {"quantity": "Oblivious baseline 1/m", "value": baseline},
        {"quantity": "Ball empirical exact-ID (matched privacy)", "value": float(ball_attack.metrics.get("exact_identification_success", np.nan))},
        {"quantity": "Standard empirical exact-ID (matched privacy)", "value": float(standard_attack.metrics.get("exact_identification_success", np.nan))},
        {"quantity": "Ball prior rank", "value": float(ball_attack.metrics.get("prior_rank", np.nan))},
        {"quantity": "Standard prior rank", "value": float(standard_attack.metrics.get("prior_rank", np.nan))},
    ]
)


def posterior_probs_from_attack(attack):
    log_scores = np.asarray(attack.diagnostics["candidate_log_scores"], dtype=np.float64)
    log_scores = log_scores - float(np.max(log_scores))
    probs = np.exp(log_scores)
    probs = probs / float(np.sum(probs))
    return probs


posterior_df = pd.DataFrame(
    {
        "candidate_id": np.arange(len(X_candidates), dtype=int),
        "source_id": candidate_source_ids,
        "distance_to_target": candidate_distances,
        "is_true_target": [sid == true_source_id for sid in candidate_source_ids],
        "ball_posterior_prob": posterior_probs_from_attack(ball_attack),
        "standard_posterior_prob": posterior_probs_from_attack(standard_attack),
    }
).sort_values("ball_posterior_prob", ascending=False).reset_index(drop=True)

print("Matched-privacy bounds (Ball vs separate standard release at the same epsilon, delta)")
display(matched_bounds_df.round(10))

print("Same-sigma geometry references (holding sigma = sigma_Ball fixed)")
display(same_sigma_bounds_df.round(10))

print("Sanity checks: under Gaussian calibration, matched-privacy gaps should be numerically tiny")
display(consistency_df.round(12))

display(attack_df.round(6))
display(posterior_df.round(6))


## 7. Visual summary

The panels are intentionally separated so that matched-privacy and same-sigma quantities are never conflated:

- **Panel A**: candidate support feasibility inside the Ball;
- **Panel B**: theorem-backed exact-identification upper bounds for the two **matched-privacy** releases;
- **Panel C**: theorem-backed **same-sigma** geometry comparison, holding `sigma = sigma_Ball` fixed;
- **Panel D**: empirical exact-identification success for the two **matched-privacy** releases, compared with the oblivious baseline `1/m`.

The optional posterior plot at the end compares the posteriors induced by the two matched-privacy releases only.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15.6, 9.0))
axes = axes.ravel()

# Panel A: candidate distances
sorted_dists = np.sort(candidate_distances)
axes[0].plot(np.arange(1, len(sorted_dists) + 1), sorted_dists, marker="o", color=BALL_COLOR)
axes[0].axhline(radius, color=BALL_COLOR, linestyle="--", linewidth=2, label=f"Ball radius = {radius:.3f}")
axes[0].set_xlabel("candidate rank (sorted by distance)")
axes[0].set_ylabel("distance to target")
axes[0].set_title("A. Finite support inside the Ball")
axes[0].legend()

# Panel B: matched-privacy bounds
x = np.arange(len(matched_bounds_df))
width = 0.36
axes[1].bar(x - width / 2, matched_bounds_df["Ball"], width=width, label="Ball", color=BALL_COLOR)
axes[1].bar(x + width / 2, matched_bounds_df["Standard matched"], width=width, label="Standard matched", color=STANDARD_COLOR)
axes[1].set_xticks(x)
axes[1].set_xticklabels(matched_bounds_df["bound_family"])
axes[1].set_ylabel("exact-ID upper bound")
axes[1].set_title("B. Matched-privacy bounds")
axes[1].legend()

# Panel C: same-sigma geometry bounds
axes[2].bar(x - width / 2, same_sigma_bounds_df["Ball"], width=width, label="Ball", color=BALL_COLOR)
axes[2].bar(x + width / 2, same_sigma_bounds_df["Standard @ sigma_Ball"], width=width, label=r"Standard @ $\sigma_{\mathrm{Ball}}$", color=NEUTRAL_COLOR)
axes[2].set_xticks(x)
axes[2].set_xticklabels(same_sigma_bounds_df["bound_family"])
axes[2].set_ylabel("exact-ID upper bound")
axes[2].set_title(r"C. Same-$\sigma$ geometry comparison")
axes[2].legend()

# Panel D: empirical exact identification vs baseline
empirical_plot = pd.DataFrame(
    {
        "quantity": ["baseline 1/m", "Ball empirical", "Standard empirical"],
        "value": [
            baseline,
            float(ball_attack.metrics.get("exact_identification_success", np.nan)),
            float(standard_attack.metrics.get("exact_identification_success", np.nan)),
        ],
    }
)
axes[3].bar(empirical_plot["quantity"], empirical_plot["value"], color=[NEUTRAL_COLOR, BALL_COLOR, STANDARD_COLOR])
axes[3].set_ylim(0.0, 1.0)
axes[3].set_ylabel("success probability")
axes[3].set_title("D. Empirical exact identification (matched privacy)")
for i, v in enumerate(empirical_plot["value"]):
    axes[3].text(i, float(v) + 0.02, f"{float(v):.3f}", ha="center", va="bottom")

plt.tight_layout()
plt.show()


# Optional: inspect the posterior concentration more closely.
topk = min(10, len(posterior_df))
plot_post = posterior_df.head(topk).copy()
plot_post["candidate_label"] = [f"cand {i}" + (" *" if t else "") for i, t in zip(plot_post["candidate_id"], plot_post["is_true_target"])]

fig, ax = plt.subplots(figsize=(8.2, 4.8))
positions = np.arange(topk)
width = 0.38
ax.bar(positions - width / 2, plot_post["ball_posterior_prob"], width=width, label="Ball", color=BALL_COLOR)
ax.bar(positions + width / 2, plot_post["standard_posterior_prob"], width=width, label="Standard matched", color=STANDARD_COLOR)
ax.set_xticks(positions)
ax.set_xticklabels(plot_post["candidate_label"], rotation=20)
ax.set_ylabel("posterior probability")
ax.set_title("Top posterior candidates under matched-privacy releases (* = true target)")
ax.legend()
plt.tight_layout()
plt.show()


## 8. Minimal paper-ready takeaways from this notebook run

For this one worked example, the key quantities to report are:

- selected Ball radius `r` and standard radius `2B`;
- Ball vs standard noise scales `sigma` at the same `(epsilon, delta)`;
- Ball vs standard test accuracy at the same `(epsilon, delta)`;
- matched-privacy direct and RDP exact-identification upper bounds;
- same-sigma standard reference bounds, explicitly labeled as geometry-only comparisons;
- empirical exact-identification success and the oblivious baseline `1/m`.

A useful sanity check is that, under Gaussian calibration, the matched-privacy Ball and matched-privacy standard direct bounds should agree up to numerical tolerance. If they do not, something in the notebook or package code is inconsistent.

The tables above are now arranged so that matched-privacy and same-sigma quantities are never mixed in the same summary.
